In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Carga de datos y exploración inicial

## Diccionario de variables

### Fichero de movimiento (`wsdr.csv`) — ventas semanales por tienda y producto

| Variable    | Descripción |
|-------------|-------------|
| `STORE`     | Número de tienda de Dominick's. |
| `UPC`       | Código de producto (Universal Product Code). Clave para unir con el catálogo. |
| `WEEK`      | Índice de semana interno de Dominick's (no es una fecha; se mapea con la tabla de semanas). |
| `MOVE`      | Unidades vendidas en esa tienda y semana. Es la demanda. |
| `QTY`       | Nº de unidades que forman el paquete de venta (p. ej. 3 en "3 por 2$"). |
| `PRICE`     | Precio de venta del paquete. El precio unitario es `PRICE / QTY`. |
| `SALE` | Código de promoción de la semana: `B` = bonus buy, `C` = cupón, `S` = rebaja simple; en blanco = precio regular. (`G` aparece en los datos pero no está en el codebook oficial → se trata como promoción no documentada. Aviso: la variable no se rellena de forma consistente, un blanco no garantiza ausencia de promoción.) |
| `PROFIT`    | Margen bruto del minorista, en %. Permite optimizar margen, no solo ingreso. |
| `OK`        | Flag de calidad del dato: 1 = fiable, 0 = posible problema. |
| `PRICE_HEX` | Precio en hexadecimal a precisión completa. Se ignora (usamos `PRICE`). |
| `PROFIT_HEX`| Margen en hexadecimal a precisión completa. Se ignora (usamos `PROFIT`). |

### Catálogo de productos (`upcsdr.csv`) — un registro por UPC

| Variable    | Descripción |
|-------------|-------------|
| `com_code`  | Código de *commodity*: clasificación interna de Dominick's que agrupa productos afines. |
| `upc`       | Código de producto. Clave para unir con el fichero de movimiento. |
| `descrip`   | Descripción del producto (marca y tipo). |
| `size`      | Tamaño o formato del envase (p. ej. "2 LTR", "12 PK"). |
| `case`      | Unidades por caja (formato de compra al por mayor). |
| `nitem`     | Número de ítems / identificador interno del producto. |

In [2]:
RAW = '../data/raw' # Ruta a los datos
mov = pd.read_csv(f'{RAW}/wsdr.csv')
upc = pd.read_csv(f'{RAW}/upcsdr.csv', encoding='latin-1')

In [3]:
print(mov.shape) # Forma de mov
mov.head()

(17730501, 11)


,STORE,UPC,WEEK,MOVE,QTY,PRICE,SALE,PROFIT,OK,PRICE_HEX,PROFIT_HEX
0,2,179,1,1,1,10.0,NaN,99.9,1,4024000000000000,4058F9999999999A
1,2,179,2,0,1,0.0,NaN,0.0,1,0000000000000000,0000000000000000
2,2,179,3,0,1,0.0,NaN,0.0,1,0000000000000000,0000000000000000
3,2,179,4,2,1,10.0,NaN,99.9,1,4024000000000000,4058F9999999999A
4,2,179,5,0,1,0.0,NaN,0.0,1,0000000000000000,0000000000000000


In [4]:
mov.info()
mov.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17730501 entries, 0 to 17730500
Data columns (total 11 columns):
 #   Column      Dtype  
---  ------      -----  
 0   STORE       int64  
 1   UPC         int64  
 2   WEEK        int64  
 3   MOVE        int64  
 4   QTY         int64  
 5   PRICE       float64
 6   SALE        object 
 7   PROFIT      float64
 8   OK          int64  
 9   PRICE_HEX   object 
 10  PROFIT_HEX  object 
dtypes: float64(2), int64(6), object(3)
memory usage: 1.5+ GB


,STORE,UPC,WEEK,MOVE,QTY,PRICE,PROFIT,OK
count,1.773050e+07,1.773050e+07,1.773050e+07,1.773050e+07,1.773050e+07,1.773050e+07,1.773050e+07,1.773050e+07
mean,8.393260e+01,4.617154e+09,2.343443e+02,1.706751e+01,1.056829e+00,1.428817e+00,1.180705e+01,9.856988e-01
std,3.676353e+01,5.256305e+09,1.116569e+02,8.765091e+01,5.609347e-01,1.865940e+00,1.965496e+01,1.187296e-01
min,2.000000e+00,1.790000e+02,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,-9.999000e+01,0.000000e+00
25%,5.600000e+01,1.690002e+09,1.510000e+02,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
50%,9.000000e+01,4.300095e+09,2.470000e+02,2.000000e+00,1.000000e+00,8.900000e-01,4.030000e+00,1.000000e+00
75%,1.140000e+02,7.020270e+09,3.310000e+02,1.000000e+01,1.000000e+00,2.000000e+00,2.465000e+01,1.000000e+00
max,1.460000e+02,7.714348e+10,3.990000e+02,9.999900e+04,2.400000e+01,1.188800e+02,9.999000e+01,1.000000e+00


In [5]:
mov.columns.tolist()
print(f'Nº tiendas únicas: {mov["STORE"].nunique()}'), print(f'Nº productos únicos: {mov["UPC"].nunique()}') # número de tiendas y productos únicos
print('-'*15)
print(f'Fecha semana mínima: {mov["WEEK"].min()}'), print(f'Fecha semana máxima: {mov["WEEK"].max()}') # valor mínimo y máximo de semanas
print('-'*15)
print(f'Calidad del dato: {mov["OK"].value_counts(dropna=False)}')
print('-'*15)
print(f'Información sobre promociones: {mov["SALE"].value_counts(dropna=False)}')
print('-'*15)
print('Valores ausentes')
mov.isna().sum()

Nº tiendas únicas: 93
Nº productos únicos: 1720
---------------
Fecha semana mínima: 1
Fecha semana máxima: 399
---------------
Calidad del dato: OK
1    17476933
0      253568
Name: count, dtype: int64
---------------
Información sobre promociones: SALE
NaN    14287219
B       1666010
S       1664212
G         73034
C         40026
Name: count, dtype: int64
---------------
Valores ausentes


STORE                0
UPC                  0
WEEK                 0
MOVE                 0
QTY                  0
PRICE                0
SALE          14287219
PROFIT               0
OK                   0
PRICE_HEX            0
PROFIT_HEX           0
dtype: int64

In [6]:
print(upc.shape)
upc.head()

(1746, 6)


,COM_CODE,UPC,DESCRIP,SIZE,CASE,NITEM
0,225,179,BLOOD GLUCOSE SCREEN,EACH,1,9990290
1,235,418,~DIET CRYSTAL PEPSI,24/12O,1,54950
2,235,419,~CRYSTAL PEPSI 24 PA,24/12O,1,54940
3,235,420,PEPSI COLA CANS,24/12O,1,54880
4,235,421,PEPSI DIET CANS,24/12O,1,54890


Tras un pequeño vistazo inicial, para la tabla `mov` vamos a eliminar las columnas con el `PRICE_HEX` y `PROFIT_HEX` para liberar espacio, ya que son columnas que no van a ser utilizadas.

In [7]:
mov = mov.drop(columns=['PRICE_HEX', 'PROFIT_HEX'])

La variable `OK` de la tabla `mov` proporciona información sobre la calidad del dato, lo que nos puede llevar a plantear la eliminación de los registros que no cumplen con la calidad del dato. Antes de eliminar, se realiza un análisis rápido de esta variable para determinar la acción a realizar.

In [8]:
flagged = mov[mov['OK'] == 0] # Seleccionamos registros que no cumplen calidad
print(flagged[['PRICE','MOVE','PROFIT']].describe())

               PRICE           MOVE         PROFIT
count  253568.000000  253568.000000  253568.000000
mean        0.611474      19.572604       6.409334
std         1.334854     412.815302      13.328842
min         0.000000       0.000000       0.000000
25%         0.000000       0.000000       0.000000
50%         0.000000       0.000000       0.000000
75%         0.690000       1.000000       2.630000
max        10.000000   99999.000000      99.990000


Se puede ver como en la columnas seleccionadas el valor es principalmente 0. Esto nos lleva a eliminar estos registros para evitar futuros problemas.

In [ ]:
mov = mov[mov['OK'] == 1].copy() # Filtramos tabla mov para quedar con registros que cumplen calidad de dato.
mov.shape

(17476933, 9)

In [26]:
print((mov["PRICE"] == 0).sum()) # Comprobamos precios
print('-'*15)
mov[(mov["PRICE"] == 0)].head()

6735190
---------------


,STORE,UPC,WEEK,MOVE,QTY,PRICE,SALE,PROFIT,OK
1,2,179,2,0,1,0.0,NaN,0.0,1
2,2,179,3,0,1,0.0,NaN,0.0,1
4,2,179,5,0,1,0.0,NaN,0.0,1
6,2,179,7,0,1,0.0,NaN,0.0,1
8,2,179,9,0,1,0.0,NaN,0.0,1


Con los datos filtrados donde cumplen la calidad del dato, seguimos teniendo precios con valor 0. Seguimos profundizando en el análisis para comprender mejor este fenómeno

In [24]:
# Miramos si hay ventas con precio 0
print(((mov["PRICE"] == 0) & (mov["MOVE"] > 0)).sum())

0


La conclusión alcanzada es que se trata de semanas en donde el producto no se ha vendido. Para evitar problemas futuros, volvemos a filtrar la tabla para quedarnos solo con registros en donde hay movimiento, es decir, en donde el producto se ha vendido.

In [27]:
mov = mov[mov['PRICE'] > 0].copy()
mov.shape

(10741743, 9)

Antes de unir las dos tablas y transformar la columna `WEEK` para crear un índice de fechas, vamos a comprobar que las tablas coinciden y no hay duplicados.

In [ ]:
# Comprobamos si hay duplicados
print(f"Valores duplicados: {mov.duplicated(subset=['STORE', 'UPC', 'WEEK']).sum()}")
print('-'*15)
# Compatibilidad de clave entre tablas
print(mov['UPC'].dtype), print(upc['UPC'].dtype) # Deben coincidir ambos tipos
print('-'*15)
print(f"Fracción de filas compatible: {mov['UPC'].isin(upc['UPC']).mean()}")
print('-'*15)
upc['UPC'].duplicated().sum() # Comprobamos presencia de valores duplicados en catálogo.

Valores duplicados: 0
---------------
int64
int64
---------------
Fracción de filas compatible: 1.0
---------------


np.int64(0)